<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Was sind KI-Agenten?
</b></font> </br></p>

---

In [ ]:
#@title  Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()


# 1 | Uebersicht
---


In diesem Modul klaeren wir die Grundlage fuer den gesamten Kurs:
1. Was ein KI-Agent genau ist
2. Wie sich Agent, Chatbot, RPA und LLM-Call unterscheiden
3. Welche Eigenschaften Agenten in der Praxis brauchbar machen
4. Warum das ReAct/TAO-Prinzip zentral ist



**Lernziel:** Sie koennen danach begruenden, wann ein Agent gerechtfertigt ist - und wann nicht.


# 2 | Definition: Agent vs. Chatbot vs. RPA
---


| Systemtyp | Staerke | Grenze | Typischer Einsatz |
|---|---|---|---|
| **LLM-Call** | Schnell, einfach | Keine Tool-Nutzung, kein Zustand | Kurzantwort, Umformulierung |
| **Chatbot** | Gute Interaktion | Meist reaktiv, wenig Prozesslogik | FAQ, Service-Dialog |
| **RPA-Bot** | Deterministische Automatisierung | Geringe Flexibilitaet bei Ausnahmen | Formular-/Backoffice-Automation |
| **KI-Agent** | Flexible Zielverfolgung mit Tools | Mehr Komplexitaet und Kosten | Recherche, Analyse, Orchestrierung |


Ein Agent kombiniert die Staerken von Sprachverarbeitung mit aktiver Handlungsfaehigkeit.


In [ ]:
def waehle_system(steps:int, tool_calls:bool, deterministic:bool)->str:
    if deterministic and not tool_calls:
        return "RPA-Bot"
    if steps <= 1 and not tool_calls:
        return "LLM-Call oder Chatbot"
    if tool_calls or steps > 1:
        return "KI-Agent"
    return "Chatbot"
beispiele = [
    (1, False, False, "Produktbeschreibung umformulieren"),
    (3, True, False, "Reiseoptionen vergleichen und buchen vorbereiten"),
    (5, False, True, "Rechnungen in ERP uebertragen"),
]
for steps, tools, deterministic, beschreibung in beispiele:
    system = waehle_system(steps, tools, deterministic)
    print(f"{beschreibung}: {system}")


# 3 | Die 4 Eigenschaften eines Agenten
---


Ein praxisreifer Agent zeigt vier Kerneigenschaften:
1. **Autonomie**: kann Teilentscheidungen selbst treffen
2. **Reaktionsfaehigkeit**: reagiert auf neue Informationen
3. **Proaktivitaet**: verfolgt ein Ziel ueber mehrere Schritte
4. **Soziale Faehigkeit**: interagiert klar mit Mensch und/oder anderen Agenten
Je mehr davon robust umgesetzt sind, desto eher sprechen wir von einem Agenten statt einem einfachen Chat-Interface.


In [ ]:
def score_agent_case(autonomie:int, reaktion:int, proaktiv:int, sozial:int)->int:
    werte = [autonomie, reaktion, proaktiv, sozial]
    assert all(0 <= x <= 3 for x in werte), "Werte zwischen 0 und 3"
    return sum(werte)
cases = {
    "FAQ-Bot": score_agent_case(0, 2, 0, 2),
    "Research-Agent": score_agent_case(3, 3, 3, 2),
    "Dokumentenklassifizierung": score_agent_case(1, 1, 1, 0),
}
for name, score in cases.items():
    label = "Agent-Kandidat" if score >= 8 else "eher kein Agent"
    print(f"{name}: {score}/12 -> {label}")


# 4 | ReAct/TAO-Prinzip
---


Das ReAct/TAO-Prinzip beschreibt den Kernzyklus agentischer Systeme:
- **Thought**: Naechsten sinnvollen Schritt planen
- **Action**: Tool oder Modellaufruf ausfuehren
- **Observation**: Ergebnis pruefen und Zustand aktualisieren      

Dieser Zyklus laeuft iterativ, bis das Ziel erreicht oder ein Abbruchkriterium greift.     

**Kostenblick:** Jeder Loop kostet Tokens, Latenz und oft API-Aufrufe. Deshalb sind Stop-Kriterien und Logging Pflicht.


In [ ]:
tao_steps = [
    ("Thought", "Ich benoetige aktuelle Wetterdaten fuer Berlin."),
    ("Action", "Tool 'get_weather' fuer Berlin aufrufen"),
    ("Observation", "Tool liefert 6 C, Regenwahrscheinlichkeit 80%"),
    ("Thought", "Ich empfehle Regenschutz und Plan B fuer Indoor-Aktivitaet."),
]
for i, (phase, text) in enumerate(tao_steps, start=1):
    mprint(f"**{i}. {phase}:** {text}")


In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  Prozessdiagramm</font> </br></p>

mermaid('''
graph TD
A[Thought] --> B[Action]
B --> C[Observation]
C --> A
A --> D[Stop, wenn Ziel erreicht]
''')


# 5 | Agent-Typen
---


| Typ | Charakteristik | Gut geeignet fuer |
|---|---|---|
| **ReAct-Agent** | Freiere Schleife aus Denken + Handeln | Offene Recherche, Diagnostik |
| **Function/Tool-Calling Agent** | Modell ruft definierte Tools gezielt auf | Integrationen mit klaren APIs |
| **Workflow-/Graph-Agent** | Strukturierte Knoten, klare Uebergaenge | Compliance, robuste Produktion |
Praxisregel:
- Starten Sie fuer Prototypen mit create_agent()
- Wechseln Sie zu LangGraph, sobald Kontrolle, Routing oder Human-in-the-Loop zentral werden


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def addiere(a: float, b: float) -> float:
    """Addiert zwei Zahlen und gibt die Summe zurueck."""
    return a + b

@tool
def ist_werktag(tag: str) -> str:
    """Gibt vereinfacht zurueck, ob ein Tag typischerweise ein Werktag ist."""
    tage = {
        "montag": True, "dienstag": True, "mittwoch": True,
        "donnerstag": True, "freitag": True,
        "samstag": False, "sonntag": False,
    }
    flag = tage.get(tag.strip().lower())
    if flag is None:
        return "Unbekannter Tag"
    return "Werktag" if flag else "Wochenende"

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[addiere, ist_werktag],
    system_prompt="Nutze Tools nur, wenn sie fuer die Frage wirklich hilfreich sind.",
)
result = agent.invoke({"messages": [{"role": "user", "content": "Ist Samstag ein Werktag und was ist 14 + 9?"}]})
mprint("## ­Agent-Antwort")
mprint(result["messages"][-1].content)

# 6 | Recap: Baseline LLM ohne Tools
---


Ein einzelner LLM-Call kann sehr gut formulieren, erklaeren und strukturieren.
Was ihm ohne Tooling fehlt:
- keine verifizierten Echtzeitdaten
- keine sicheren Aktionen in externen Systemen
- keine robuste Mehrschritt-Orchestrierung
Genau diese Luecke schliessen Agenten mit Tool-Calling und Schleifenlogik.


In [ ]:
# LangChain
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein hilfreicher Assistent."),
    ("human", "{user_input}")
])

# LLM initialisieren (Kurznotation: "provider:model")
llm = init_chat_model("openai:gpt-4o-mini")

result = llm.invoke("Warum ist ein reiner LLM-Call fuer komplexe Prozessautomatisierung oft nicht ausreichend? Nenne 3 Gruende.")

mprint("## ­LLM-Baseline")
mprint('---')
mprint(result.content)

# A | Aufgabe
---


Bearbeiten Sie eine Mini-Analyse aus Ihrem Arbeitskontext:


<p><font color='black' size="5">
Use-Case-Check: Agent oder nicht?
</font></p>


Waehlen Sie einen realen Prozess (z. B. Ticket-Analyse, Angebotspruefung, Recherche-Workflow) und entscheiden Sie begruendet:
- Reicht ein LLM-Call?
- Reicht ein Chatbot?
- Brauchen Sie einen KI-Agenten?


**Optionale Teilaufgaben:**
- Beschreiben Sie den geplanten TAO-Zyklus in 3-5 Schritten
- Definieren Sie 2 konkrete Tools, die der Agent benoetigt
